# PyTorch 零基础 3/6：Tensor 运算、广播与维度变换

这是第 1～41 课 ASR 主线之前的桥梁课。先预测，再运行；看懂输出后必须改一个值验证自己的解释。

| 项目 | 内容 |
|---|---|
| 前置要求 | 完成基础 2；能解释任意示例 Tensor 的每个轴 |
| 建议投入 | 60～90 分钟，可分两次完成 |
| 核心概念 | 逐元素运算与归约、广播规则、reshape、transpose 与矩阵乘法 |
| 完成标准 | 能解释代码、独立完成练习、从空白重写本课核心函数 |


## 课前诊断（先不要运行代码）

1. 用自己的话解释：逐元素运算与归约。
2. 猜测 广播规则 最容易出现哪一种错误。
3. 写下你对 reshape、transpose 与矩阵乘法 的暂时理解；不会可以明确写“不知道”。

这三题不计分，只用于留下学习前证据。


## 1. 逐元素运算与归约


In [ ]:
import torch

x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("x * 2 =\n", x * 2)
print("全部均值：", x.mean())
print("每行均值：", x.mean(dim=1))
print("每列最大值：", x.max(dim=0).values)

assert x.mean(dim=1).shape == (2,)
assert torch.allclose(x.mean(dim=1), torch.tensor([2.0, 5.0]))


## 2. 广播：从尾部维度对齐

两个维度相等或其中一个为 1 时可以广播。能运行不代表语义正确，所以先写 shape 再写算式。


In [ ]:
features = torch.arange(2 * 4 * 3, dtype=torch.float32).reshape(2, 4, 3)
feature_bias = torch.tensor([10.0, 20.0, 30.0])
shifted = features + feature_bias

print(features.shape, "+", feature_bias.shape, "->", shifted.shape)
assert shifted.shape == (2, 4, 3)
assert torch.equal(shifted[0, 0], features[0, 0] + feature_bias)


## 3. reshape 改分组方式，transpose/permute 改轴顺序


In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
flat_time = x.reshape(2, 12)
swapped = x.transpose(1, 2)
conv_layout = x.permute(0, 2, 1)

print("原始 [B,T,F]：", x.shape)
print("reshape：", flat_time.shape)
print("transpose：", swapped.shape)
print("Conv1d 常用 [B,F,T]：", conv_layout.shape)

assert swapped.shape == (2, 4, 3)
assert torch.equal(swapped, conv_layout)


## 4. 矩阵乘法把最后一个特征轴映射到新维度


In [ ]:
B, T, F, H = 2, 5, 3, 4
features = torch.randn(B, T, F)
weight = torch.randn(F, H)
encoded = features @ weight

print("[B,T,F] @ [F,H] ->", encoded.shape)
assert encoded.shape == (B, T, H)


## 5. Mask 也依赖广播


In [ ]:
values = torch.tensor([[[1.0], [2.0], [99.0]], [[3.0], [99.0], [99.0]]])
mask = torch.tensor([[True, True, False], [True, False, False]])
masked = values.masked_fill(~mask.unsqueeze(-1), 0.0)

print(masked.squeeze(-1))
assert torch.equal(masked.squeeze(-1), torch.tensor([[1.0, 2.0, 0.0], [3.0, 0.0, 0.0]]))


## 本课练习（保留作答区）


1. `x.shape=[2,3,4]` 时，`x.mean(dim=1).shape` 是什么？
2. `[2,5,80] + [80]` 为什么能广播？
3. 判断 `[2,5,80] + [5]` 是否能广播，并运行验证。
4. 把 `[B,T,F]` 转成 Conv1d 所需 `[B,F,T]`。
5. 解释 reshape 与 permute 的区别。
6. 写一个 `[B,T,F] @ [F,H]` 示例并断言输出 shape。
7. 故意制造矩阵乘法维度不匹配，读懂报错中的 shape。
8. 实现按最后一维做零均值标准化，注意数值稳定性。
9. 比较 `mean()`、`mean(dim=1)` 和 `mean(dim=(1,2))`。
10. 解释为什么广播错误有时不会报错却会悄悄改变语义。


评分：每题 0～2 分。达到 16/20 可以继续；12～15 分次日重做错题；低于 12 分回看代码并从空白复现。


## 离场票与间隔复习

- [ ] 我能闭卷解释：逐元素运算与归约、广播规则、reshape、transpose 与矩阵乘法。
- [ ] 我能预测核心代码的 shape、dtype 或数值方向。
- [ ] 我能从空白重写至少一个函数，并通过正常、边界、错误输入测试。
- [ ] 我能说出一个“代码能运行但语义错误”的例子。

复习安排：明天闭卷回忆 5 分钟；7 天后重做第 4、7、10 题；30 天后重新构造最小实验。

下一步：基础 4：autograd、loss、梯度与优化器。
